In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

# ── Fonts — no LaTeX required ─────────────────────────────────────────────────
plt.rcParams.update({
    "text.usetex":        False,
    "font.family":        "serif",
    "font.serif":         ["Palatino", "Palatino Linotype", "Georgia",
                           "Times New Roman", "DejaVu Serif"],
    "mathtext.fontset":   "cm",
    "axes.labelsize":     11,
    "axes.titlesize":     11,
    "xtick.labelsize":    9,
    "ytick.labelsize":    9,
    "figure.dpi":         150,
})

# ── Thesis colour palette ─────────────────────────────────────────────────────
TROPICALRAINFOREST = (0.0, 0.46, 0.37)
ITALICRED          = (177/255, 56/255, 69/255)
LIGHTGREY     = (0.65, 0.65, 0.65)

def _pale(rgb, mix=0.72):
    return tuple(c + (1 - c) * mix for c in rgb)

# ── ERW simulator ─────────────────────────────────────────────────────────────
def simulate_erw(n, p, q=0.5, rng=None):
    if rng is None:
        rng = np.random.default_rng()
    xi = np.empty(n + 1, dtype=np.int8)
    xi[0] = 0
    xi[1] = 1 if rng.random() < q else -1
    for t in range(1, n):
        past = xi[rng.integers(0, t) + 1]
        xi[t + 1] = past if rng.random() < p else -past
    S = np.empty(n + 1, dtype=np.int64)
    S[0] = 0
    S[1:] = np.cumsum(xi[1:])
    return S

def counting_zeros(S):
    return np.cumsum(S == 0)

# ── Parameters ────────────────────────────────────────────────────────────────
N_STEPS = 20000
N_PATHS = 250
N_SHOW  = 20
Q       = 0.5

REGIMES = [
    (0.80, r"$p = 0.80$", TROPICALRAINFOREST),
    (0.90, r"$p = 0.90$", ITALICRED),
]

steps = np.arange(N_STEPS + 1)
rng   = np.random.default_rng()

# ── Simulate ──────────────────────────────────────────────────────────────────
all_Z = []
for p_val, _, _ in REGIMES:
    Z_paths = []
    for _ in range(N_PATHS):
        S = simulate_erw(N_STEPS, p=p_val, q=Q, rng=rng)
        Z_paths.append(counting_zeros(S))
    all_Z.append(np.array(Z_paths))

# ── Plot ──────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(5.6, 2.9), sharey=False)
fig.subplots_adjust(left=0.10, right=0.985, bottom=0.18, top=0.88, wspace=0.34)

for ax, (p_val, label, color), Z in zip(axes, REGIMES, all_Z):
    mean_Z = Z.mean(axis=0)

    idx = rng.choice(len(Z), size=min(N_SHOW, len(Z)), replace=False)
    for j in idx:
        ax.plot(
            steps, Z[j],
            color=_pale(color, 0.72),
            lw=0.65,
            alpha=0.72,
            rasterized=True
        )

    ax.plot(steps, mean_Z, color=color, lw=1.35)

    ax.set_title(label, fontsize=10, pad=5, color=LIGHTGREY)
    ax.set_xlabel(r"$n$", labelpad=3)

    ax.set_xlim(0, N_STEPS)
    ax.set_ylim(0, 200)

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_linewidth(0.55)
    ax.spines["bottom"].set_linewidth(0.55)
    ax.tick_params(width=0.55, length=3, direction="out")

    ax.xaxis.set_major_locator(ticker.FixedLocator([0, N_STEPS // 2, N_STEPS]))
    ax.xaxis.set_major_formatter(
        ticker.FuncFormatter(
            lambda x, _:
                r"$0$" if x == 0 else
                r"$10000$" if x == N_STEPS // 2 else
                r"$20000$"
        )
    )
    ax.yaxis.set_major_locator(ticker.MaxNLocator(nbins=4, integer=True))

axes[0].set_ylabel(r"$Z_n$", labelpad=4)

plt.savefig("/home/claude/fig_transience_zeros.png", dpi=300, bbox_inches="tight")
plt.show()
print("Done.")